# PyTorch

CPU에서 전체 학습 루프를 실행하고 validation이 parameter를 바꾸지 않는지 확인합니다. torch가 없으면 같은 계산을 NumPy로 실행합니다.

핵심 셀은 Python 3 + NumPy만 사용합니다. 데이터 다운로드와 API 키가 필요하지 않습니다.

## 같은 입력 계약

In [ ]:
import numpy as np
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.],[2.,1.],[1.,2.]])
y=(2*X[:,0:1]+3*X[:,1:2]+1)
assert X.shape==(6,2) and y.shape==(6,1)

## NumPy forward·backward·update

In [ ]:
w=np.zeros((2,1)); b=np.zeros((1,)); losses=[]
for step in range(250):
    pred=X@w+b
    error=pred-y
    loss=np.mean(error**2)
    grad_w=2*X.T@error/len(X)
    grad_b=2*error.mean(axis=0)
    w-=.05*grad_w; b-=.05*grad_b
    losses.append(loss)
print("first and last loss",losses[0],losses[-1])
assert losses[-1]<.001

## validation과shape실패 검사

In [ ]:
before=w.copy()
X_valid=np.array([[.5,.5],[1.5,.5]])
y_valid=2*X_valid[:,0:1]+3*X_valid[:,1:2]+1
valid_loss=np.mean((X_valid@w+b-y_valid)**2)
assert np.array_equal(w,before)
print("validation loss",valid_loss)
print("wrong residual shape",(X@w-y.ravel()).shape)

## 과제

- NumPy loop에서 gradient factor 2를 제거하면 같은 learning rate에서 수렴 속도가 어떻게 바뀌나요?
- validation 전에 weight를 저장하고 validation 뒤 정확히 같은지 검사하세요.

## 해답과 검증

In [ ]:
saved=w.copy(); _=np.mean((X_valid@w+b-y_valid)**2)
assert np.array_equal(saved,w)
print("validation leaves parameters unchanged")
assert (X@w-y.ravel()).shape==(6,6)

def trace(factor, lr, steps=40):
    wt=np.zeros((2,1)); bt=np.zeros((1,)); history=[]
    for _ in range(steps):
        err=X@wt+bt-y
        history.append(float(np.mean(err**2)))
        wt -= lr*factor*(X.T@err/len(X))
        bt -= lr*factor*err.mean(axis=0)
    return np.array(history), wt, bt

full,wf,bf=trace(2,.05)
half,wh,bh=trace(1,.05)
matched,wm,bm=trace(1,.10)
assert full[-1] < half[-1]
assert np.allclose(full,matched)
assert np.allclose(wf,wm) and np.allclose(bf,bm)
print("factor 2 / factor 1 losses", full[-1], half[-1])
print("Doubling the learning rate restores the same updates")


## 선택: 실제 프레임워크

설치되어 있을 때만 실행합니다. 설치 명령이나 네트워크 접근은 자동으로 실행하지 않습니다.

In [ ]:
import importlib.util
if importlib.util.find_spec("torch"):
    import torch
    torch.manual_seed(7)
    xt=torch.tensor(X,dtype=torch.float32); yt=torch.tensor(y,dtype=torch.float32)
    model=torch.nn.Linear(2,1)
    opt=torch.optim.SGD(model.parameters(),lr=.05)
    history=[]
    for step in range(300):
        model.train(); opt.zero_grad(); pred=model(xt)
        assert pred.shape==yt.shape
        loss=torch.nn.functional.mse_loss(pred,yt)
        loss.backward(); opt.step(); history.append(loss.item())
    snap=[p.detach().clone() for p in model.parameters()]
    model.eval()
    with torch.no_grad():
        vl=torch.nn.functional.mse_loss(model(torch.tensor(X_valid,dtype=torch.float32)),torch.tensor(y_valid,dtype=torch.float32))
    assert all(torch.equal(a,b) for a,b in zip(snap,model.parameters()))
    assert history[-1]<history[0]
    print("torch final train/validation",history[-1],vl.item())
else:
    print("torch unavailable; the full NumPy forward/backward/update/validation loop passed")